In [3]:
!pip install chromadb
!pip install -U -q "google-genai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.4 MB/s e

In [7]:
from google.colab import userdata
from google import genai

# El cliente de Gemini para hacer los embedding
GEMINI_API_KEY = userdata.get('GOOGLE_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

## API de TMDB para conseguir información de películas y cast

In [167]:
import json
import requests

# Para obtener información de una película
# usamos la API de TMDB
TMDB_API_KEY = userdata.get('TMDB_KEY')
TMDB_HEADERS = {
      "accept": "application/json",
      "Authorization": f"Bearer {TMDB_API_KEY}"
}

def get_movies_info(title):
  url = f"https://api.themoviedb.org/3/search/movie?query={title}&include_adult=false&language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_reviews(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_cast(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_person_details(person_id):
  url = f"https://api.themoviedb.org/3/person/{person_id}"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)


## Añadir películas a la colección "movies" de ChromaDB

In [156]:
import chromadb
chroma_client = chromadb.PersistentClient(path="chroma_db")

# Función para añadir películas encontradas a la colección
# la colección es una parte de la base de datos
# hay colecciones por categorías (películas, actores, reviews...)
def add_movies_to_collection(movies_info):
  movies_col = chroma_client.get_or_create_collection('movies')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  def get_cast_info(movie):
    cast_found = get_movie_cast(str(movie['id']))
    director = "";
    cast = "";

    for person in cast_found['cast']:
      if person['known_for_department'] == 'Directing':
        director += person['name'] + ", "
      else:
        cast += person['name'] + ", "
    return director, cast

  def get_metadata(movie, director, cast):
    return {
        "movie_title" : movie['title'],
        "director"    : director,
        "cast"        : cast,
        "popularity"  : movie['popularity'],
        "release_date": movie['release_date'],
        "vote_average": movie['vote_average'],
        "vote_count"  : movie['vote_count']
    }

  for movie in movies_info['results']:
    # Consultamos con nuestra colección
    result = movies_col.get(
      ids=[str(movie['id'])],
      include=[]
    )

    # Si no existe, la añade
    if not result['ids'] and movie['overview']:
      # Primero obtenemos al cast
      director, cast = get_cast_info(movie)

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=movie['overview']).embeddings[0]
      ids.append(str(movie['id']))
      embeddings.append(content_embeddings.values)
      metadatas.append(get_metadata(movie, director, cast))
      documents.append(movie['overview'])

  if len(ids) > 0:
    movies_col.add(
        ids = ids,
        embeddings = embeddings,
        metadatas = metadatas,
        documents = documents
    )

  print(f"Added {len(ids)} movies.")

# Ejemplo de cómo usarlo junto a la búsqueda en TMDB
movies = get_movies_info('mario the movie')
add_movies_to_collection(movies)

Added 5 movies.


## Añadir reviews a la colección "reviews" de ChromaDB

In [177]:
def add_reviews_to_collection(movie_id):
  reviews_col = chroma_client.get_or_create_collection('reviews')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  reviews = get_movie_reviews(movie_id)

  def get_metadata(review):
    if review["author_details"]["rating"]:
      return {
          "author"  : review["author"],
          "rating"  : review["author_details"]["rating"]
      }
    else:
      return {
          "author"  : review["author"],
      }

  for review in reviews['results']:

    # Consultamos con nuestra colección
    result = reviews_col.get(
      ids=[str(review['id'])],
      include=[]
    )
    # Si no existe, la añade
    if not result['ids'] and review['content']:
      ids.append(review['id'])
      metadatas.append(get_metadata(review))

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=review['content']).embeddings[0]
      embeddings.append(content_embeddings.values)
      documents.append(review['content'])

      if len(ids) > 0:
        reviews_col.add(
          ids = ids,
          embeddings = embeddings,
          metadatas = metadatas,
          documents = documents
        )

  print(f"Added {len(ids)} reviews.")

# Ejemplo con la película 99 de TMDB (Todo Sobre Mi Madre)
add_reviews_to_collection("99")


Added 0 reviews.


## Añadir actores a la colección "people" de ChromaDB

In [180]:
def add_person_to_collection(person_id):
  people_col = chroma_client.get_or_create_collection('people')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  details = get_person_details(person_id)

  # Consultamos con nuestra colección
  result = people_col.get(
    ids=[str(details['id'])],
    include=[]
  )

  if not result['ids'] and details['biography']:
    ids.append(str(details['id']))

    gender = "Not set"
    if details['gender'] == 1:
      gender = "female"
    elif details['gender'] == 2:
      gender = "male"
    elif details['gender'] == 3:
      gender = "non binary"

    metadatas.append({'name': details['name'], 'department': details['known_for_department'], 'gender': gender})
    content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                      contents=details['biography']).embeddings[0]
    embeddings.append(content_embeddings.values)
    documents.append(details['biography'])

    if len(ids) > 0:
      people_col.add(
      ids = ids,
      embeddings = embeddings,
      metadatas = metadatas,
      documents = documents
    )

    print(f"Added {details['name']}")
  else:
    print("Person already exists in collection")

# Ejemplo con persona 31 (Tom Hanks)
add_person_to_collection("31")


Added Tom Hanks


## Ejemplo de consulta a la colección "movies" de ChromaDB

In [166]:
# Ejemplo de hacerle una pregunta a la colección
query = "película sobre coches"

# Hay que hacer un embedding porque hemos no usamos el modelo
# nativo de chromadb, sino el de gemini al meterlos en la colección
query_embedding = client.models.embed_content(model="gemini-embedding-001",
                                              contents=query).embeddings[0]
movies_col = chroma_client.get_or_create_collection('movies')

res = movies_col.query(
    query_embeddings=[query_embedding.values],
    n_results=3
)

res['metadatas'][0]

[{'cast': 'Mateusz Janicki, Sandra Drzymalska, Maria Dębska, Jacek Beler, Anna Dymna, Olgierd Blecharz, Kalina Kowalczuk, Piotr Sega, Przemysław Bluszcz, Ewa Błaszczyk, Adam Ferency, Radosław Rożniecki, Eryk Pratsko, Juliusz Godzina, Krzysztof Ogonek, Szymon Kukla, Mateusz Łasowski, Kajetan Miros, Magdalena Majtyka, Krzysztof Satała, Andrzej Szubski, Joanna Król, Grzegorz Szypka, Anna Rokita, Michał Kasprzak-Komarczewski, Dariusz Maj, Mateusz Grydlik, Tomasz Augustynowicz, Igor Górewicz, Michał Petrow, Piotr Stefańczuk, India Maag, Aron Jończyk, Łukasz Lewandowski, Rafał Meusiak, Mariusz Majuch, Mila Majewska, Danuta Burska, Jan Niemczyk, Aleksandra Żebrowska, Bożena Kulig-Fiallo, Jerzy Kornacki, Agata Sierakiewicz, Rafał Maj, Zbigniew Rowiński, Joanna Stępień, Kamil Płocki, Arturs Abramenko, ',
  'vote_average': 5.806,
  'release_date': '2023-07-12',
  'director': '',
  'movie_title': 'Mr. Car and the Knights Templar',
  'popularity': 2.4609,
  'vote_count': 152},
 {'vote_average': 6.